# Assignment 3: Local Association Matrix 

**Student names**: Eskild Sæther <br>
**Group number**: 101 <br>
**Date**: 12.10.2026

## Important notes
Please read and follow these rules. Submissions that do not fulfill them may be returned.
1. You may work in groups of maximum 2 students.
2. Submit in **.ipynb** format only.
3. The assignment must be typed. Handwritten answers are not accepted.

**Due date**: 16.10.2026 23:59

### What you will do 
- Build a **local association matrix** from Cranfield collection.
- Compute the **normalized association matrix**.
- Use the normalized matrix to **identify neighborhood terms** for expansion for given queries.


---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**


### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.

In [6]:
# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"./cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)

Parsed 1400 documents.


## 3.1  Local association matrix

For the given Cranfield document collection in cran.all.1400 construct a local association matrix to identify association clusters. Use the docs variable with the parsed file. Omit stopwords in the STOPWORDS list given below from the vocabulary. 


The correlation factors $c_{u,v}$ between any pair of terms $w_u$ and $w_v$ are defined as  
$c_{u,v} = \sum_{d_j \in D} f_{u,j} \cdot f_{v,j}$  

$f_{u,j}$ is the raw term frequency of $w_u$ in document $d_j$.

### Weighting variants: **scalar** and **metric**

Add two alternative weighting schemes for the matrix (only the formula for assigning the matrix cell value changes):

- **Metric weighting** :
Let $w_u(n,j)$ and $w_v(m,j)$ denote the $n$-th and $m$-th occurrences of terms $w_u$ and $w_v$ in document $d_j$.  
Define a distance function $r(w_u(n,j), w_v(m,j))$ (e.g., $r(i,k) = 1 + |i - k|$).  
Then:

$$
c_{u,v} = \sum_{d_j \in D} \sum_n \sum_m \frac{1}{r(w_u(n,j), w_v(m,j))}
$$


- **Scalar weighting** :
Let $\vec{s}_u = \langle c_{u,x_1}, c_{u,x_2}, \dots, c_{u,x_n} \rangle$ be the neighborhood vector of term $w_u$, and similarly for $w_v$.  
Then:

$$
c_{u,v} = \frac{\vec{s}_u \cdot \vec{s}_v}{|\vec{s}_u| \cdot |\vec{s}_v|}
$$

In [7]:
# TODO: Construct a local association matrix for the cranfield collection. Use both weighting variants.
import math

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

#Not sure if i should remove numbers, they may be useful for correlation. for example bm25.
removal = set(['.', ',', ';', '(', ')', ':', '/', '\\'])
# Your code here
#Skal lage association matrix for hele collection 1400 docs, jeg skal lage associaton
#matrix for terms.
#Føler tokens burde være en dict med antall kopier av hvert ord.
for doc in docs.values():
    temporary_tokens = []
    doc['tokens'] = {}
    doc['title'] = doc['title'].lower()
    doc['abstract'] = doc['abstract'].lower()
    
    for char in removal:
        doc['title'] = doc['title'].replace(char, "")
        doc ['abstract'] = doc['abstract'].replace(char, "")
    temporary_tokens = doc['title'].split()
    #Major discovery av extend
    temporary_tokens.extend(doc['abstract'].split())
    #Kanskje det blir rot fordi jeg modifiserer listen jeg går i gjennom.
    new_temporary_tokens = []
    for token in temporary_tokens:
        if token not in STOPWORDS:
            new_temporary_tokens.append(token)
    temporary_tokens = new_temporary_tokens

    #Trenger en liste også
    for token in temporary_tokens:
        doc['token_list'] = temporary_tokens
    #Gjør om til set for å få alle unike
    for token in set(temporary_tokens):
        doc['tokens'][token] = temporary_tokens.count(token)

    #print(doc['tokens'])

#først lager vi term frequency matrix.
#det blir basically en ny dict med key: term, of value: amount
tf = {}
for doc in docs.values():
    for term, amount in doc['tokens'].items():
        if(term not in tf.keys()):
            tf[term] = amount
        else:
            tf[term] += amount
print(tf)

#skal nå lage association matrix. basically kolonner og rader er terms, og verdien
#i den matrisen er korrelansen
#for å løse dette kan ha dict{term: dict{related term: score}} Hvor jeg har mange related term
# association_matrix = {}
# #Tar lowkirkuenly 7 sekunder å kjøre.....
# ##########################################
# #kommentert ut association_matrix fordi den tar lagn tid + burde kanskje fikse den
# #så den kjører i rimelig tid
# for term in tf.keys():
#     association_matrix[term] = {}
#     for related_term in tf.keys():
#         association_matrix[term][related_term] = 0


#Hvorfor er jeg så iq sperra på dette problemet....
print(docs[1]['tokens'].items())

association_matrix = {}
for doc in docs.values():
    for term, term_count in doc['tokens'].items():
        if (term not in association_matrix.keys()):
            association_matrix[term] = {}

        for related_term, related_count in doc['tokens'].items():
            if (related_term not in association_matrix[term].keys()):
                association_matrix[term][related_term] = related_count * term_count
            else:
                association_matrix[term][related_term] += term_count * related_count

print("weight verdi for comparative association_matrix", association_matrix['comparative'])

# print("token list test", docs[1]['token_list'])

#Skal nå lage metric weighting, problemet er at jeg trenger indeksen til termsa.
#Så jeg trenger lowkirkuenly en vanlig liste med tokens og ikke en dict.
metric_weight_matrix = {}

#Er det en doc som ikke har token_list ??? For en rar bug
for doc in docs.values():
    if 'token_list' not in doc.keys():
        continue
    for term, index in zip(doc['token_list'], range(0, len(doc['token_list']))):
        for related_term, related_index in zip(doc['token_list'], range(0, len(doc['token_list']))):
            if(term not in metric_weight_matrix.keys()):
                metric_weight_matrix[term] = {}
            if(related_term not in metric_weight_matrix[term].keys()):
                metric_weight_matrix[term][related_term] = 0

            distance_r = 1 + abs(index-related_index)
            metric_weight_matrix[term][related_term] += 1 / distance_r

print("metric weight for comparative", metric_weight_matrix['comparative'])

##okay tror det er gucci, skal nå lage scalar weightning, mp først finne ut av hva det er
#Skal jeg bruke metric_weight_matrix eller similarity matrix for radene....
#Velger å bruke similarity matrix.
scalar_weight_matrix = {}
#Her må vi iterer gjennom hver rad av vår similarity index

#This took about  20 minutes to run

# for term in association_matrix.keys():
#     scalar_weight_matrix[term] = {}
#     term_row = association_matrix[term].keys()

#     term_row_length = 0
#     for value in association_matrix[term].values():
#         term_row_length+= value*value
#     term_row_length = math.sqrt(term_row_length)

#     for related_term in association_matrix[term].keys():
#         scalar_weight_matrix[term][related_term] = 0

#         row_multiplication_sum = 0
#         related_row_length = 0
#         for related_term_related_terms in association_matrix[related_term].keys():
#             if related_term_related_terms in term_row:
#                 v2 = association_matrix[related_term][related_term_related_terms]
#                 v1 = association_matrix[term][related_term_related_terms]
#                 row_multiplication_sum += v1*v2
#             related_row_length += v2*v2
#         related_row_length = math.sqrt(related_row_length)

#         scalar_score = row_multiplication_sum / (term_row_length*related_row_length)
#         scalar_weight_matrix[term][related_term] = scalar_score


# print("scalar weight for comparative", scalar_weight_matrix['comparative'])

# print("scalar weight of itself", scalar_weight_matrix['comparative']['comparative'])

{'evaluation': 33, 'increase': 133, 'increment': 7, 'experimental': 466, 'problem': 391, 'destalling': 5, 'remaining': 8, 'treatments': 12, 'stream': 233, 'ratios': 151, 'supporting': 9, 'showed': 53, 'configuration': 61, 'intended': 12, 'specific': 71, 'results': 896, 'together': 42, 'boundary-layer-control': 1, 'spanwise': 45, 'well': 167, 'basis': 77, 'subtracting': 2, 'integrated': 33, 'evidence': 28, 'flow': 2052, 'due': 180, 'distribution': 409, 'comparative': 9, 'span': 42, 'angles': 176, 'potential': 79, 'theory': 857, 'experiment': 93, 'wing': 486, 'produced': 45, 'determine': 115, 'substantial': 18, 'study': 203, 'loading': 137, 'free': 196, 'aerodynamics': 30, 'part': 133, 'lift': 314, 'agree': 33, 'investigation': 356, 'effect': 458, 'effects': 597, 'made': 448, 'different': 126, 'propeller': 62, 'order': 168, 'velocity': 496, 'attack': 225, 'curves': 92, 'slipstream': 42, 'found': 422, 'empirical': 27, 'theoretical': 308, 'somewhat': 23, 'can': 414, 'paper': 236, 'flat': 2

## 3.2 Normalized association matrix

Compute the normalized association matrix from the unnormalized matrix computed above. 

To normalize the matrix use the following formula: <br>
$c'_{u,v} = \frac{c_{u,v}}{c_{u,u} + c_{v,v} - c_{u,v}}$  


In [11]:
#TODO: Compute the normalized association matrix 

# Your code here
#normalisert association matrix. C (u,u) er ikke det bare antall appearnces av terms totalt...

normalized_association_matrix = {}

for term in association_matrix:
    normalized_association_matrix[term] = {}
    term_scalar = association_matrix[term][term]
    for related_term in association_matrix[term]:
        related_term_scalar = association_matrix[related_term][related_term]
        assoc_value = association_matrix[term][related_term]

        normalized_association_matrix[term][related_term] = assoc_value / (term_scalar + related_term_scalar - assoc_value)

print("Value of normalized assoc matrix for comparative", normalized_association_matrix['comparative'])
print("value of comparative compared to comparative", normalized_association_matrix['comparative']['comparative'])

Value of normalized assoc matrix for comparative {'evaluation': 0.029411764705882353, 'increase': 0.00398406374501992, 'increment': 0.06666666666666667, 'experimental': 0.006749156355455568, 'problem': 0.0011074197120708748, 'destalling': 0.1111111111111111, 'remaining': 0.041666666666666664, 'treatments': 0.03333333333333333, 'stream': 0.005385996409335727, 'ratios': 0.02981029810298103, 'supporting': 0.04, 'showed': 0.009708737864077669, 'configuration': 0.0070921985815602835, 'intended': 0.03571428571428571, 'specific': 0.014285714285714285, 'results': 0.003331482509716824, 'together': 0.016129032258064516, 'boundary-layer-control': 0.058823529411764705, 'spanwise': 0.007751937984496124, 'well': 0.017857142857142856, 'basis': 0.008849557522123894, 'subtracting': 0.05555555555555555, 'integrated': 0.018867924528301886, 'evidence': 0.022727272727272728, 'flow': 0.0005805852299117511, 'due': 0.006349206349206349, 'distribution': 0.001893939393939394, 'comparative': 1.0, 'span': 0.00925

## 3.3 Neighborhood terms

With the help of the normalized local association matrix, identify the neighborhood terms that should be used for expansion for the following queries (queries_assignment3):


In [12]:
# Do not change this code
queries_assignment3 = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [18]:
#TODO: Identify neighborhood terms for queries_assignment3

# Your code here
#How many neighborhood terms should i find? let's do top 5

for query in queries_assignment3:
    related_queries_score = {}
    for token in query.split():
        if token not in normalized_association_matrix.keys():
            continue
        for related_term, score in normalized_association_matrix[token].items():
            if related_term == token:
                continue
            if related_term not in related_queries_score.keys():
                related_queries_score[related_term] = score
            else:
                related_queries_score[related_term] +=score

    top_5 = sorted(related_queries_score.items(), key=lambda kv: kv[1], reverse=True)[:5]

    print(query)
    for term, score in top_5:
        print(f"  {term}: {score:.3f}")


gas pressure
  number: 0.351
  mach: 0.334
  flow: 0.301
  results: 0.259
  shock: 0.229
structural aeroelastic flight high speed aircraft
  may: 0.417
  speeds: 0.416
  aerodynamic: 0.402
  discussed: 0.401
  made: 0.399
heat conduction composite slabs
  slab: 0.756
  transfer: 0.662
  refractory: 0.597
  medium: 0.502
  shielded: 0.444
boundary layer control
  boundary: 0.710
  layer: 0.707
  laminar: 0.668
  flow: 0.391
  turbulent: 0.373
compressible flow nozzle
  pressure: 0.339
  number: 0.334
  laminar: 0.321
  layer: 0.317
  boundary: 0.311
combustion chamber injection
  products: 0.549
  one-foot: 0.415
  high-enthalpy: 0.355
  ignition: 0.344
  combustion-heated: 0.341
laminar turbulent transition
  layer: 0.658
  boundary: 0.646
  reynolds: 0.603
  layers: 0.434
  number: 0.418
fatigue crack growth
  life: 0.310
  structural: 0.210
  s: 0.173
  behaviour: 0.150
  random: 0.150
wing tip vortices
  vortex: 0.485
  trailing: 0.370
  combinations: 0.296
  downwash: 0.294
  wings